In [2]:
from pathlib import Path
import pandas as pd
import shutil

# ============================================================
# 1. 경로 설정
# ============================================================

BASE_DIR = Path(r"E:\jyp\ct_data_2d")

# 위에서 만든 CSV 파일을 여기에 넣어두고 사용
CSV_PATH = BASE_DIR / "luna16_normal_candidates_by_subset_files.csv"

# 정상 후보만 따로 모을 출력 폴더
OUT_DIR = BASE_DIR / "LUNA16_Normal_Candidates_287"

# True  = 실제 복사 안 함, 검사만 함
# False = 실제 복사 실행
DRY_RUN = False

# 정상 후보 개수 검증용
EXPECTED_NORMAL_COUNT = 287


# ============================================================
# 2. 기본 검사
# ============================================================

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR가 존재하지 않습니다: {BASE_DIR}")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"정상 후보 CSV 파일이 없습니다: {CSV_PATH}\n"
        f"luna16_normal_candidates_by_subset_files.csv 파일을 {BASE_DIR} 안에 넣어주세요."
    )

df = pd.read_csv(CSV_PATH)

required_cols = {"subset", "seriesuid", "mhd_filename", "raw_filename"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(
        f"CSV에 필요한 컬럼이 없습니다: {missing_cols}\n"
        f"필요 컬럼: {required_cols}\n"
        f"현재 컬럼: {list(df.columns)}"
    )

# 공백 제거
for col in ["subset", "seriesuid", "mhd_filename", "raw_filename"]:
    df[col] = df[col].astype(str).str.strip()

# 중복 검사
duplicated = df[df["seriesuid"].duplicated(keep=False)]
if len(duplicated) > 0:
    raise ValueError("seriesuid 중복이 있습니다. CSV를 다시 확인해야 합니다.")

# 개수 검사
if len(df) != EXPECTED_NORMAL_COUNT:
    raise ValueError(
        f"정상 후보 개수가 예상과 다릅니다.\n"
        f"예상: {EXPECTED_NORMAL_COUNT}개\n"
        f"현재 CSV: {len(df)}개"
    )

print(f"[확인] 정상 후보 CSV 로드 완료: {len(df)}개")


# ============================================================
# 3. subset 폴더와 .mhd / .raw 존재 검사
# ============================================================

missing_files = []
copy_plan = []

for _, row in df.iterrows():
    subset_name = row["subset"]
    seriesuid = row["seriesuid"]
    mhd_filename = row["mhd_filename"]
    raw_filename = row["raw_filename"]

    subset_dir = BASE_DIR / subset_name

    src_mhd = subset_dir / mhd_filename
    src_raw = subset_dir / raw_filename

    dst_subset_dir = OUT_DIR / subset_name
    dst_mhd = dst_subset_dir / mhd_filename
    dst_raw = dst_subset_dir / raw_filename

    if not subset_dir.exists():
        missing_files.append({
            "seriesuid": seriesuid,
            "problem": "subset_folder_missing",
            "path": str(subset_dir)
        })
        continue

    if not src_mhd.exists():
        missing_files.append({
            "seriesuid": seriesuid,
            "problem": "mhd_missing",
            "path": str(src_mhd)
        })

    if not src_raw.exists():
        missing_files.append({
            "seriesuid": seriesuid,
            "problem": "raw_missing",
            "path": str(src_raw)
        })

    if src_mhd.exists() and src_raw.exists():
        copy_plan.append({
            "subset": subset_name,
            "seriesuid": seriesuid,
            "src_mhd": src_mhd,
            "src_raw": src_raw,
            "dst_subset_dir": dst_subset_dir,
            "dst_mhd": dst_mhd,
            "dst_raw": dst_raw
        })

print(f"[확인] 복사 가능 정상 후보: {len(copy_plan)}개")
print(f"[확인] 누락 문제 개수: {len(missing_files)}개")

if missing_files:
    missing_df = pd.DataFrame(missing_files)
    missing_report_path = BASE_DIR / "missing_normal_candidate_files.csv"
    missing_df.to_csv(missing_report_path, index=False, encoding="utf-8-sig")

    print("\n[중단] 누락된 파일이 있습니다.")
    print(f"누락 목록 저장 위치: {missing_report_path}")
    display(missing_df.head(20))

    raise FileNotFoundError("누락 파일이 있어서 복사를 중단했습니다.")


# ============================================================
# 4. DRY RUN 결과 확인
# ============================================================

summary_df = (
    pd.DataFrame(copy_plan)
    .groupby("subset")
    .size()
    .reset_index(name="normal_candidate_count")
)

print("\n[subset별 정상 후보 개수]")
display(summary_df)

print("\n[복사 예시 5개]")
for item in copy_plan[:5]:
    print(item["src_mhd"], "->", item["dst_mhd"])
    print(item["src_raw"], "->", item["dst_raw"])


# ============================================================
# 5. 실제 복사
# ============================================================

if DRY_RUN:
    print("\n[DRY_RUN=True]")
    print("아직 실제 복사는 하지 않았습니다.")
    print("위 결과가 맞으면 DRY_RUN = False 로 바꾸고 다시 실행하세요.")
else:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    copied_records = []

    for item in copy_plan:
        item["dst_subset_dir"].mkdir(parents=True, exist_ok=True)

        shutil.copy2(item["src_mhd"], item["dst_mhd"])
        shutil.copy2(item["src_raw"], item["dst_raw"])

        copied_records.append({
            "subset": item["subset"],
            "seriesuid": item["seriesuid"],
            "mhd_path": str(item["dst_mhd"]),
            "raw_path": str(item["dst_raw"])
        })

    copied_df = pd.DataFrame(copied_records)
    copied_report_path = OUT_DIR / "copied_normal_candidates_287.csv"
    copied_df.to_csv(copied_report_path, index=False, encoding="utf-8-sig")

    print("\n[완료] 정상 후보 복사 완료")
    print(f"출력 폴더: {OUT_DIR}")
    print(f"복사 개수: {len(copied_df)}개")
    print(f"복사 보고서: {copied_report_path}")

[확인] 정상 후보 CSV 로드 완료: 287개
[확인] 복사 가능 정상 후보: 287개
[확인] 누락 문제 개수: 0개

[subset별 정상 후보 개수]


,subset,normal_candidate_count
0,subset0,22
1,subset1,28
2,subset2,33
3,subset3,24
4,subset4,27
5,subset5,35
6,subset6,26
7,subset7,35
8,subset8,28
9,subset9,29



[복사 예시 5개]
E:\jyp\ct_data_2d\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd -> E:\jyp\ct_data_2d\LUNA16_Normal_Candidates_287\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd
E:\jyp\ct_data_2d\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.raw -> E:\jyp\ct_data_2d\LUNA16_Normal_Candidates_287\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.raw
E:\jyp\ct_data_2d\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.mhd -> E:\jyp\ct_data_2d\LUNA16_Normal_Candidates_287\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.mhd
E:\jyp\ct_data_2d\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.raw -> E:\jyp\ct_data_2d\LUNA16_Normal_Candidates_287\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.raw
E:\jyp\ct_data_2d\subset0\1.3.6.1.4.1.14519.5.2.1.6279.6001.126121460017257137098781143514.mhd -> E:

In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"E:\jyp\ct_data_2d")
CSV_PATH = BASE_DIR / "luna16_normal_candidates_by_subset_files.csv"
OUT_DIR = BASE_DIR / "LUNA16_Normal_Candidates_287"

df = pd.read_csv(CSV_PATH)

for col in ["subset", "seriesuid", "mhd_filename", "raw_filename"]:
    df[col] = df[col].astype(str).str.strip()

missing_after_copy = []

for _, row in df.iterrows():
    subset_dir = OUT_DIR / row["subset"]

    mhd_path = subset_dir / row["mhd_filename"]
    raw_path = subset_dir / row["raw_filename"]

    if not mhd_path.exists():
        missing_after_copy.append({
            "seriesuid": row["seriesuid"],
            "missing": "mhd",
            "path": str(mhd_path)
        })

    if not raw_path.exists():
        missing_after_copy.append({
            "seriesuid": row["seriesuid"],
            "missing": "raw",
            "path": str(raw_path)
        })

mhd_files = list(OUT_DIR.rglob("*.mhd"))
raw_files = list(OUT_DIR.rglob("*.raw"))

print(f"[확인] 복사된 .mhd 개수: {len(mhd_files)}개")
print(f"[확인] 복사된 .raw 개수: {len(raw_files)}개")
print(f"[확인] 누락 파일 개수: {len(missing_after_copy)}개")

if missing_after_copy:
    missing_df = pd.DataFrame(missing_after_copy)
    display(missing_df)
    raise FileNotFoundError("복사 후 누락 파일이 있습니다.")
else:
    print("[완료] 정상 후보 287개가 .mhd/.raw 쌍으로 모두 복사되었습니다.")

[확인] 복사된 .mhd 개수: 287개
[확인] 복사된 .raw 개수: 287개
[확인] 누락 파일 개수: 0개
[완료] 정상 후보 287개가 .mhd/.raw 쌍으로 모두 복사되었습니다.
